# Chapter 7 | Generative Tutoring and Personalised Pedagogy

**Companion notebook:** dialogue analysis, contextual prompt design, pedagogical text features, semantic alignment and student-turn sentiment.

**Source:** [Education Dialogue Dataset, Google Research](https://github.com/google-research-datasets/Education-Dialogue-Dataset). The repository describes **simulated teacher–student dialogues**, produced with Gemini Ultra, rather than observations collected from actual classrooms. It lists 40,000 training conversations and 7,234 evaluation conversations. The repository has been archived and is read-only. The original field names include the spellings `student_prefrences` and `teacher_prefrences`.

**Interpretation:** Descriptive differences in synthetic dialogues are not evidence of learning gains, clinical states or the causal effectiveness of a tutoring strategy. The optional matched-response demonstration, when enabled, produces *new* outputs rather than reproducing historical experiments.

**Setup:** Python 3.10+; `pip install pandas numpy matplotlib seaborn scikit-learn textblob`.
For optional sentence embeddings: `pip install sentence-transformers`. For local response generation, a compatible Ollama service and locally installed model are required.

Run cells from top to bottom. Charts are saved as PDF and PNG under `outputs/chapter07/`. The dataset remains separate from the notebook.

<div style="border:1px solid #d8e0ea;border-left:4px solid #4c6b8a;border-radius:7px;background:#f7f9fc;padding:11px 15px;margin:14px 0 8px 0;">
<strong>01 | Imports and reproducibility</strong><br><span style="color:#45556a;">Load the analysis libraries and define a deterministic sample for computationally intensive steps.</span>
</div>

In [ ]:
from pathlib import Path
from collections import Counter
from itertools import islice
import json
import os
import re
import textwrap
import urllib.request
import urllib.error
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import paired_cosine_distances

SEED = 42
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 130, "savefig.dpi": 250,
                     "font.size": 10, "axes.titlesize": 12})
sns.set_theme(style="whitegrid", context="notebook")
OUTPUT_DIR = Path("outputs/chapter07")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Evaluation conversations are the default; loading all training files is optional.
LOAD_TRAIN_FILES = False
AUTO_DOWNLOAD_EVALUATION = True
MAX_TEXT_DIALOGUES = 1200
RUN_SENTENCE_EMBEDDINGS = False
RUN_LOCAL_COMPARISON = False
LOCAL_MODEL = os.environ.get("TUTOR_MODEL", "")

def save_figure(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(OUTPUT_DIR / f"{name}.{ext}", bbox_inches="tight")
    plt.show()

print("Output directory:", OUTPUT_DIR.resolve())

<div style="border:1px solid #d8e0ea;border-left:4px solid #4c6b8a;border-radius:7px;background:#f7f9fc;padding:11px 15px;margin:14px 0 8px 0;">
<strong>02 | Dataset discovery</strong><br><span style="color:#45556a;">Locate the original JSON files; download the evaluation split from the cited repository only if it is missing.</span>
</div>

In [ ]:
SOURCE_URL = "https://github.com/google-research-datasets/Education-Dialogue-Dataset"
DATA_FILES = ["conversations_eval.json"]
if LOAD_TRAIN_FILES:
    DATA_FILES += [f"conversations_train{i}.json" for i in range(1, 6)]

candidates = [
    Path("data/Education-Dialogue-Dataset"),
    Path("data/Education-Dialogue-Dataset-main"),
    Path("Education-Dialogue-Dataset"),
    Path("Education-Dialogue-Dataset-main"),
    Path("data"),
    Path("."),
]
DATA_DIR = next((p for p in candidates if (p / "conversations_eval.json").is_file()), candidates[0])
DATA_DIR.mkdir(parents=True, exist_ok=True)

if not (DATA_DIR / "conversations_eval.json").is_file() and AUTO_DOWNLOAD_EVALUATION:
    remote = "https://raw.githubusercontent.com/google-research-datasets/Education-Dialogue-Dataset/main/conversations_eval.json"
    print("Fetching evaluation data from the source repository...")
    try:
        urllib.request.urlretrieve(remote, DATA_DIR / "conversations_eval.json")
    except (urllib.error.URLError, OSError) as exc:
        raise FileNotFoundError(
            "The evaluation JSON could not be downloaded. Download conversations_eval.json "
            "from the dataset repository and place it under data/Education-Dialogue-Dataset/."
        ) from exc
missing = [name for name in DATA_FILES if not (DATA_DIR / name).is_file()]
if missing:
    raise FileNotFoundError("Missing source data files: " + ", ".join(missing))
print("Dataset folder:", DATA_DIR.resolve())

<div style="border:1px solid #d8e0ea;border-left:4px solid #4c6b8a;border-radius:7px;background:#f7f9fc;padding:11px 15px;margin:14px 0 8px 0;">
<strong>03 | Load and validate conversations</strong><br><span style="color:#45556a;">Read the repository’s JSON records, check the required fields and preserve dataset split identifiers.</span>
</div>

In [ ]:
def load_records(directory, file_names):
    records = []
    for file_name in file_names:
        with (directory / file_name).open("r", encoding="utf-8") as handle:
            payload = json.load(handle)
        if isinstance(payload, dict):
            payload = payload.get("data", payload.get("conversations", []))
        if not isinstance(payload, list):
            raise ValueError(f"Expected a JSON list in {file_name}")
        split = "evaluation" if "eval" in file_name else "training"
        for idx, entry in enumerate(payload):
            if not isinstance(entry, dict):
                continue
            if not isinstance(entry.get("background_info"), dict) or not isinstance(entry.get("conversation"), list):
                continue
            records.append({"dialogue_id": f"{file_name}:{idx:06d}", "split": split,
                            "background_info": entry["background_info"],
                            "conversation": entry["conversation"]})
        print(f"{file_name}: {len(payload):,} source records")
    if not records:
        raise ValueError("No valid dialogue records found")
    return records

records = load_records(DATA_DIR, DATA_FILES)
print(f"Total valid conversations: {len(records):,}")
print("Background fields:", sorted(records[0]["background_info"].keys()))
print("Turn fields:", sorted(records[0]["conversation"][0].keys()))

<div style="border:1px solid #d8e0ea;border-left:4px solid #4c6b8a;border-radius:7px;background:#f7f9fc;padding:11px 15px;margin:14px 0 8px 0;">
<strong>04 | Construct dialogue and turn tables</strong><br><span style="color:#45556a;">Create separate tables for conversation-level metadata and individual teacher or student utterances.</span>
</div>

In [ ]:
def background_value(bg, *names):
    for name in names:
        if name in bg and bg[name] is not None:
            return str(bg[name])
    return ""

def normalise_role(role):
    role = str(role).strip().lower()
    if any(k in role for k in ("teacher", "tutor", "instructor")):
        return "teacher"
    if any(k in role for k in ("student", "learner", "pupil")):
        return "student"
    return role or "unknown"

rows, turns = [], []
for record in records:
    bg = record["background_info"]
    did = record["dialogue_id"]
    turn_list = record["conversation"]
    rows.append({
        "dialogue_id": did, "split": record["split"],
        "topic": background_value(bg, "topic"),
        "student_preference": background_value(bg, "student_prefrences", "student_preferences"),
        "teacher_preference": background_value(bg, "teacher_prefrences", "teacher_preferences"),
        "student_reaction": background_value(bg, "student_reactions"),
        "teacher_reaction": background_value(bg, "teacher_reactions"),
        "dialogue_turns": len(turn_list),
        "word_count": sum(len(str(t.get("text", "")).split()) for t in turn_list if isinstance(t, dict)),
    })
    last_student = ""
    student_idx = 0
    for pos, turn in enumerate(turn_list):
        if not isinstance(turn, dict):
            continue
        role = normalise_role(turn.get("role", ""))
        content = str(turn.get("text", ""))
        if role == "student":
            student_idx += 1
        turns.append({"dialogue_id": did, "turn_index": pos + 1,
                      "student_turn_index": student_idx if role == "student" else np.nan,
                      "role": role, "text": content, "preceding_student_text": last_student})
        if role == "student":
            last_student = content

dialogues = pd.DataFrame(rows)
turns_df = pd.DataFrame(turns)
print(dialogues[["topic", "student_preference", "teacher_preference", "dialogue_turns"]].head(3).to_string(index=False))
print("Speaker counts:", turns_df["role"].value_counts().to_dict())

<div style="border:1px solid #d8e0ea;border-left:4px solid #4c6b8a;border-radius:7px;background:#f7f9fc;padding:11px 15px;margin:14px 0 8px 0;">
<strong>05 | Dataset overview</strong><br><span style="color:#45556a;">Summarise the dialogue lengths and document how much of the source collection has been loaded.</span>
</div>

In [ ]:
print(dialogues.groupby("split")["dialogue_turns"].agg(["count", "mean", "median", "min", "max"]).round(2))
fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.2))
sns.histplot(data=dialogues, x="dialogue_turns", bins=25, ax=axes[0], color="#4b789b")
axes[0].set(title="Dialogue length", xlabel="Recorded turns", ylabel="Conversations")
subject_counts = dialogues["topic"].value_counts().head(10).sort_values()
subject_counts.plot.barh(ax=axes[1], color="#679a91")
axes[1].set(title="Most frequent topics", xlabel="Conversations", ylabel="")
fig.tight_layout()
save_figure(fig, "source_dialogue_overview")

<div style="border:1px solid #d8e0ea;border-left:4px solid #4c6b8a;border-radius:7px;background:#f7f9fc;padding:11px 15px;margin:14px 0 8px 0;">
<strong>06 | Dialogue persistence (Section 7.4.1)</strong><br><span style="color:#45556a;">Compare recorded dialogue lengths across coarse categories derived from the scenario metadata; these are not measured emotional states.</span>
</div>

In [ ]:
def reaction_category(text):
    s = str(text).lower()
    if re.search(r"frustrat|annoy|angry|upset|rude|irritat", s):
        return "Frustration-related scenario"
    if re.search(r"disengag|bored|give up|quit|stop listening|lose interest", s):
        return "Disengagement-related scenario"
    if re.search(r"adapt|adjust|accept|flexib|keep trying", s):
        return "Adaptation-related scenario"
    return "Other scripted reactions"

dialogues["reaction_group"] = dialogues["student_reaction"].map(reaction_category)
print(dialogues["reaction_group"].value_counts().to_string())
fig, ax = plt.subplots(figsize=(10.2, 4.8))
order = dialogues.groupby("reaction_group")["dialogue_turns"].median().sort_values().index
sns.boxplot(data=dialogues, x="reaction_group", y="dialogue_turns", order=order,
            color="#92b5c9", showfliers=False, ax=ax)
ax.set(title="Figure 7.2 companion: dialogue persistence by scenario",
       xlabel="Scripted reaction description", ylabel="Recorded conversation turns")
ax.tick_params(axis="x", rotation=12)
fig.tight_layout()
save_figure(fig, "figure_7_2_observed_dialogue_length")

<div style="border:1px solid #d8e0ea;border-left:4px solid #4c6b8a;border-radius:7px;background:#f7f9fc;padding:11px 15px;margin:14px 0 8px 0;">
<strong>07 | Teaching and learning preferences (Section 7.4.2)</strong><br><span style="color:#45556a;">Compare mean recorded turns across frequently observed preference combinations; the heatmap is descriptive, not causal.</span>
</div>

In [ ]:
top_teachers = dialogues["teacher_preference"].value_counts().head(6).index
top_students = dialogues["student_preference"].value_counts().head(6).index
subset = dialogues[dialogues["teacher_preference"].isin(top_teachers) &
                   dialogues["student_preference"].isin(top_students)].copy()
means = subset.pivot_table(index="teacher_preference", columns="student_preference",
                           values="dialogue_turns", aggfunc="mean")
counts = subset.pivot_table(index="teacher_preference", columns="student_preference",
                            values="dialogue_turns", aggfunc="count").reindex_like(means)
min_group_size = 3
means = means.mask(counts < min_group_size)
short = lambda items: [textwrap.shorten(str(s), width=30, placeholder="…") for s in items]
fig, ax = plt.subplots(figsize=(10.6, 5.7))
sns.heatmap(means, annot=True, fmt=".1f", cmap="YlGnBu", ax=ax,
            cbar_kws={"label": "Mean dialogue turns"})
ax.set_xticklabels(short(means.columns), rotation=30, ha="right")
ax.set_yticklabels(short(means.index), rotation=0)
ax.set(title="Figure 7.3 companion: preference contexts and dialogue length",
       xlabel="Student preference metadata", ylabel="Teacher preference metadata")
fig.tight_layout()
save_figure(fig, "figure_7_3_preference_contexts")

<div style="border:1px solid #d8e0ea;border-left:4px solid #4c6b8a;border-radius:7px;background:#f7f9fc;padding:11px 15px;margin:14px 0 8px 0;">
<strong>08 | Contextual prompt construction (Section 7.3)</strong><br><span style="color:#45556a;">Create two explicit instructional conditions from the same topic, learner preference and prior dialogue.</span>
</div>

In [ ]:
def recent_dialogue(turn_list, max_turns=5):
    lines = []
    for item in turn_list[-max_turns:]:
        who = normalise_role(item.get("role", "speaker")).title()
        text = re.sub(r"\s+", " ", str(item.get("text", ""))).strip()
        lines.append(f"{who}: {text[:350]}")
    return "\n".join(lines)

def build_prompt(record, condition="Socratic"):
    bg = record["background_info"]
    topic = background_value(bg, "topic")
    preference = background_value(bg, "student_prefrences", "student_preferences")
    reaction = background_value(bg, "student_reactions")
    source_turns = record["conversation"]
    student_positions = [i for i, t in enumerate(source_turns)
                         if normalise_role(t.get("role", "")) == "student"]
    if not student_positions:
        raise ValueError("A learner turn is required to construct a response prompt")
    # Stop at the most recent learner contribution; do not reveal the following tutor turn.
    history = recent_dialogue(source_turns[:student_positions[-1] + 1])
    context = (f"Topic: {topic}\nStudent learning preference: {preference}\n"
               f"Scenario reaction description: {reaction}\nRecent conversation:\n{history}\n")
    if condition == "Socratic":
        instruction = ("Act as a supportive mathematics or subject tutor. Acknowledge relevant "
                       "learner concerns without inferring a clinical state. Give one concise "
                       "hint and end with one guiding question. Avoid revealing the full solution.")
    elif condition == "Direct":
        instruction = ("Act as a clear subject tutor. Give a concise explanation of the "
                       "topic and an example when helpful. Do not make claims about the learner's feelings.")
    else:
        raise ValueError("Condition must be 'Socratic' or 'Direct'")
    return instruction + "\n\n" + context + "\nTutor response:"

print(build_prompt(records[0], "Socratic")[:1800])

<div style="border:1px solid #d8e0ea;border-left:4px solid #4c6b8a;border-radius:7px;background:#f7f9fc;padding:11px 15px;margin:14px 0 8px 0;">
<strong>09 | Pedagogical text features</strong><br><span style="color:#45556a;">Compute transparent linguistic indicators; they are exploratory proxies and do not constitute validated instructional-quality scores.</span>
</div>

In [ ]:
SUPPORT_TERMS = ("understand", "let's", "together", "take your time", "can you", "good question")
ACTION_TERMS = ("try", "consider", "check", "compare", "explain", "think", "explore", "notice")
INQUIRY_TERMS = ("why", "how", "what", "which", "could you", "can you")

def text_features(response):
    text = str(response).strip()
    lower = text.lower()
    tokens = re.findall(r"\b[\w'-]+\b", lower)
    n = len(tokens)
    return {
        "Question use": float("?" in text),
        "Supportive wording": float(any(term in lower for term in SUPPORT_TERMS)),
        "Inquiry wording": float(any(re.search(r"\b" + re.escape(term) + r"\b", lower) for term in INQUIRY_TERMS)),
        "Actionable cue": float(any(re.search(r"\b" + re.escape(term) + r"\b", lower) for term in ACTION_TERMS)),
        "Concise (20–60 words)": float(20 <= n <= 60),
        "Word count": n,
    }

example_teacher = turns_df.loc[turns_df["role"].eq("teacher"), "text"].iloc[0]
print("Example teacher turn:", example_teacher[:300])
print("Text features:", text_features(example_teacher))

<div style="border:1px solid #d8e0ea;border-left:4px solid #4c6b8a;border-radius:7px;background:#f7f9fc;padding:11px 15px;margin:14px 0 8px 0;">
<strong>10 | Observed instructional wording (Section 7.4.3)</strong><br><span style="color:#45556a;">Describe linguistic features in the recorded teacher turns. No model-to-model comparison is claimed without paired generated responses.</span>
</div>

In [ ]:
selected_ids = set(dialogues["dialogue_id"].sample(n=min(len(dialogues), MAX_TEXT_DIALOGUES), random_state=SEED))
teacher_turns = turns_df[turns_df["dialogue_id"].isin(selected_ids) &
                         turns_df["role"].eq("teacher")].copy()
feature_rows = teacher_turns["text"].map(text_features)
feat = pd.DataFrame(feature_rows.tolist(), index=teacher_turns.index)
feature_columns = [c for c in feat.columns if c != "Word count"]
mean_features = feat[feature_columns].mean().sort_values()
fig, ax = plt.subplots(figsize=(9.5, 4.6))
mean_features.plot.barh(color="#537d9b", ax=ax)
ax.set(xlim=(0, 1), xlabel="Proportion of recorded teacher turns",
       title="Figure 7.4 companion: descriptive instructional text features")
fig.tight_layout()
save_figure(fig, "figure_7_4_observed_text_features")
print("Teacher turns evaluated:", len(teacher_turns))

<div style="border:1px solid #d8e0ea;border-left:4px solid #4c6b8a;border-radius:7px;background:#f7f9fc;padding:11px 15px;margin:14px 0 8px 0;">
<strong>11 | Context–response pair construction</strong><br><span style="color:#45556a;">Match each recorded teacher turn to the immediately preceding learner contribution within its conversation.</span>
</div>

In [ ]:
pairs = turns_df[
    turns_df["dialogue_id"].isin(selected_ids) &
    turns_df["role"].eq("teacher") &
    turns_df["preceding_student_text"].astype(str).str.len().gt(5) &
    turns_df["text"].astype(str).str.len().gt(5)
][["dialogue_id", "preceding_student_text", "text"]].copy()
pairs = pairs.sample(n=min(len(pairs), 1500), random_state=SEED).reset_index(drop=True)
if len(pairs) < 5:
    raise ValueError("Not enough valid teacher–student pairs for alignment analysis")
print("Observed paired turns:", len(pairs))

<div style="border:1px solid #d8e0ea;border-left:4px solid #4c6b8a;border-radius:7px;background:#f7f9fc;padding:11px 15px;margin:14px 0 8px 0;">
<strong>12 | Context–response alignment (Section 7.4.4)</strong><br><span style="color:#45556a;">Estimate paired text similarity and compare it with a shuffled-context reference. The reference is not a separate tutoring system.</span>
</div>

In [ ]:
contexts = pairs["preceding_student_text"].tolist()
responses = pairs["text"].tolist()
shuffled_indices = rng.permutation(len(pairs))

if RUN_SENTENCE_EMBEDDINGS:
    from sentence_transformers import SentenceTransformer
    encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    vectors = encoder.encode(contexts + responses, normalize_embeddings=True,
                             batch_size=64, show_progress_bar=False)
    left, right = vectors[:len(pairs)], vectors[len(pairs):]
    observed_sim = np.einsum("ij,ij->i", left, right)
    reference_sim = np.einsum("ij,ij->i", left[shuffled_indices], right)
    similarity_label = "Sentence-embedding cosine similarity"
    method_name = "sentence_embeddings"
else:
    # Explicit lexical approximation when the optional embedding model is disabled.
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=14000,
                                 stop_words="english")
    matrix = vectorizer.fit_transform(contexts + responses)
    left, right = matrix[:len(pairs)], matrix[len(pairs):]
    observed_sim = 1 - paired_cosine_distances(left, right)
    reference_sim = 1 - paired_cosine_distances(left[shuffled_indices], right)
    similarity_label = "TF–IDF cosine similarity (lexical proxy)"
    method_name = "tfidf_proxy"

fig, ax = plt.subplots(figsize=(9.5, 4.7))
plot_df = pd.DataFrame({
    "score": np.r_[observed_sim, reference_sim],
    "comparison": (["Observed conversation pair"] * len(observed_sim) +
                   ["Shuffled-context reference"] * len(reference_sim))
})
sns.histplot(data=plot_df, x="score", hue="comparison", bins=32,
             stat="density", common_norm=False, element="step", ax=ax)
ax.set(title="Figure 7.5 companion: recorded dialogue context–response alignment",
       xlabel=similarity_label, ylabel="Estimated density")
fig.tight_layout()
save_figure(fig, f"figure_7_5_observed_alignment_{method_name}")
print(plot_df.groupby("comparison")["score"].agg(["count", "mean", "median"]).round(3))

<div style="border:1px solid #d8e0ea;border-left:4px solid #4c6b8a;border-radius:7px;background:#f7f9fc;padding:11px 15px;margin:14px 0 8px 0;">
<strong>13 | Student-turn sentiment (Section 7.4.4)</strong><br><span style="color:#45556a;">Compute a text-based polarity proxy along observed student turns; the scores do not measure psychological recovery.</span>
</div>

In [ ]:
try:
    from textblob import TextBlob
except ImportError as exc:
    raise ImportError("Install textblob with: pip install textblob") from exc

student_turns = turns_df[turns_df["dialogue_id"].isin(selected_ids) &
                         turns_df["role"].eq("student")].copy()
student_turns["polarity"] = student_turns["text"].map(
    lambda s: float(TextBlob(str(s)).sentiment.polarity))
student_turns = student_turns.merge(dialogues[["dialogue_id", "reaction_group"]],
                                  on="dialogue_id", how="left")
student_turns = student_turns[student_turns["student_turn_index"].between(1, 7)].copy()
trend = (student_turns.groupby(["reaction_group", "student_turn_index"], as_index=False)
         .agg(mean_polarity=("polarity", "mean"), n=("polarity", "size")))
trend = trend[trend["n"] >= 5]
fig, ax = plt.subplots(figsize=(9.4, 4.8))
for group, sub in trend.groupby("reaction_group"):
    ax.plot(sub["student_turn_index"], sub["mean_polarity"], marker="o", label=group)
ax.axhline(0, color="#777", linewidth=0.8)
ax.set(xlabel="Student contribution index", ylabel="Mean lexical polarity (−1 to +1)",
       ylim=(-1, 1), title="Figure 7.6 companion: descriptive student-turn polarity")
ax.legend(fontsize=8, frameon=True)
fig.tight_layout()
save_figure(fig, "figure_7_6_observed_student_polarity")
print("Polarity values describe written turns in simulated dialogues, not observed learner well-being.")

### Optional matched response study

The source collection does **not** supply paired outputs from two tutoring configurations. To create a new, traceable comparison, the following cells may query a locally available model with both prompt conditions for the **same** conversation contexts. They are disabled by default. The outputs are demonstration data and must not be presented as the numerical results already reported in the chapter.

Set `RUN_LOCAL_COMPARISON = True` and `LOCAL_MODEL` to the identifier of a model installed on your local Ollama service. No student input is transmitted to a third-party API by these cells.

<div style="border:1px solid #d8e0ea;border-left:4px solid #4c6b8a;border-radius:7px;background:#f7f9fc;padding:11px 15px;margin:14px 0 8px 0;">
<strong>14 | Optional local response generation</strong><br><span style="color:#45556a;">Generate paired Socratic and direct-instruction responses for the same source contexts when explicitly enabled.</span>
</div>

In [ ]:
import urllib.request

def request_local_response(prompt, model, endpoint="http://localhost:11434/api/chat"):
    payload = {"model": model, "messages": [{"role": "user", "content": prompt}],
               "stream": False, "options": {"temperature": 0}}
    request = urllib.request.Request(endpoint,
         data=json.dumps(payload).encode("utf-8"),
         headers={"Content-Type": "application/json"}, method="POST")
    with urllib.request.urlopen(request, timeout=120) as response:
        result = json.load(response)
    return str(result["message"]["content"]).strip()

paired_responses_path = OUTPUT_DIR / "paired_responses_new_experiment.csv"
if RUN_LOCAL_COMPARISON:
    if not LOCAL_MODEL.strip():
        raise ValueError("Set LOCAL_MODEL to an installed local model identifier first")
    experimental_rows = []
    for case in records[:min(10, len(records))]:
        for condition in ("Socratic", "Direct"):
            response = request_local_response(build_prompt(case, condition), LOCAL_MODEL)
            experimental_rows.append({
                "dialogue_id": case["dialogue_id"], "condition": condition,
                "model": LOCAL_MODEL, "response": response,
                "context": recent_dialogue(case["conversation"]),
            })
    pd.DataFrame(experimental_rows).to_csv(paired_responses_path, index=False)
    print("New response study saved:", paired_responses_path)
else:
    print("Local response generation is disabled; no comparative performance claim is made.")

<div style="border:1px solid #d8e0ea;border-left:4px solid #4c6b8a;border-radius:7px;background:#f7f9fc;padding:11px 15px;margin:14px 0 8px 0;">
<strong>15 | Optional matched comparison</strong><br><span style="color:#45556a;">Describe surface features of genuinely produced paired responses; these measures require independent expert validation.</span>
</div>

In [ ]:
if paired_responses_path.exists():
    comparison = pd.read_csv(paired_responses_path)
    required = {"dialogue_id", "condition", "response", "context"}
    if not required.issubset(comparison.columns):
        raise ValueError("Response file is missing required columns: " + str(required - set(comparison.columns)))
    counts = comparison.groupby(["dialogue_id", "condition"]).size().unstack(fill_value=0)
    valid = counts.index[(counts.get("Socratic", 0) == 1) & (counts.get("Direct", 0) == 1)]
    comparison = comparison[comparison["dialogue_id"].isin(valid)].copy()
    if comparison.empty:
        print("No matched response pairs are available.")
    else:
        features = pd.DataFrame(comparison["response"].map(text_features).tolist(),
                                index=comparison.index)
        comparison = pd.concat([comparison, features], axis=1)
        profile = (comparison.groupby("condition")[["Question use", "Supportive wording",
                  "Inquiry wording", "Actionable cue", "Concise (20–60 words)"]].mean().T)
        fig, ax = plt.subplots(figsize=(8.8, 4.8))
        profile.plot.barh(ax=ax)
        ax.set(xlim=(0, 1), xlabel="Proportion of responses",
               title="New experiment: descriptive response indicators")
        fig.tight_layout()
        save_figure(fig, "optional_new_paired_response_features")
        print("New matched cases:", len(valid))
else:
    print("No matched generated responses available; comparison skipped.")

<div style="border:1px solid #d8e0ea;border-left:4px solid #4c6b8a;border-radius:7px;background:#f7f9fc;padding:11px 15px;margin:14px 0 8px 0;">
<strong>16 | Provenance and output summary</strong><br><span style="color:#45556a;">Save a compact record of the observed sample sizes, analysis methods and figure names.</span>
</div>

In [ ]:
summary = {
    "chapter": 7,
    "dataset": SOURCE_URL,
    "source_type": "simulated teacher-student conversations",
    "loaded_json_files": DATA_FILES,
    "dialogues_loaded": int(len(dialogues)),
    "turns_loaded": int(len(turns_df)),
    "text_analysis_dialogues": int(len(selected_ids)),
    "teacher_turns_scored": int(len(teacher_turns)),
    "context_response_pairs": int(len(pairs)),
    "alignment_method": method_name,
    "polarity_method": "TextBlob lexical polarity",
    "local_response_generation_enabled": bool(RUN_LOCAL_COMPARISON),
    "reproduces_historical_model_comparison": False,
    "limitations": [
        "Conversations are simulated rather than recorded classroom interactions.",
        "Text features are exploratory proxies rather than validated pedagogical measures.",
        "Context–response similarity does not measure teaching effectiveness.",
        "Turn-level lexical polarity does not establish changes in student well-being.",
    ],
}
(OUTPUT_DIR / "analysis_summary.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(summary, indent=2, ensure_ascii=False))

### Notes for readers

- **Figures 7.2 and 7.3:** descriptive summaries based on the source conversations. Differences in dialogue length cannot establish engagement, instructional effectiveness or causal effects.
- **Figure 7.4 companion:** observed teacher-turn linguistic indicators, not a measured comparison between tutor configurations.
- **Figure 7.5 companion:** within-dialogue text alignment versus shuffled-context comparison; the optional TF–IDF mode is lexical and is not equivalent to sentence-embedding similarity.
- **Figure 7.6 companion:** polarity of text in simulated student turns, not evidence of emotional rehabilitation or an intervention effect.
- **Response-generation option:** produces newly observed paired outputs only after explicit activation; compare them using expert review before making claims about pedagogical quality.
- **Research use:** the repository currently exposes no explicit licence file. Check permissions before redistributing its underlying JSON data. Cite the source repository and its associated research paper where appropriate.